## Crawilng and parsing API Documentation

In [1]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from typing import Set, List

BASE_URL = "https://wbsg-uni-mannheim.github.io/PyDI/"

def is_internal_link(url: str) -> bool:
    return url.startswith(BASE_URL)

def clean_text(soup: BeautifulSoup) -> str:
    # remove navigation, footer, sidebar
    for tag in soup(["nav", "footer", "header", "script", "style"]):
        tag.decompose()

    main = soup.find("div", class_="document")
    if not main:
        return ""

    return main.get_text(separator="\n", strip=True)


def crawl_pydi_docs(start_url: str) -> List[str]:
    visited: Set[str] = set()
    to_visit = [start_url]
    documents = []

    while to_visit:
        url = to_visit.pop()
        if url in visited:
            continue

        print(f"[*] Crawling: {url}")
        visited.add(url)

        try:
            resp = requests.get(url, timeout=10)
            soup = BeautifulSoup(resp.text, "html.parser")

            text = clean_text(soup)
            if text:
                documents.append(text)

            for link in soup.find_all("a", href=True):
                href = urljoin(url, link["href"])
                href = href.split("#")[0]  # remove anchors

                if is_internal_link(href) and href not in visited:
                    to_visit.append(href)

        except Exception as e:
            print(f"[x] Failed: {url} ({e})")

    return documents


In [2]:
docs = crawl_pydi_docs(BASE_URL)

[*] Crawling: https://wbsg-uni-mannheim.github.io/PyDI/
[*] Crawling: https://wbsg-uni-mannheim.github.io/PyDI/api/PyDI.html
[*] Crawling: https://wbsg-uni-mannheim.github.io/PyDI/api/PyDI.utils.html
[*] Crawling: https://wbsg-uni-mannheim.github.io/PyDI/_modules/PyDI/utils/similarity_registry.html
[*] Crawling: https://wbsg-uni-mannheim.github.io/PyDI/_modules/index.html
[*] Crawling: https://wbsg-uni-mannheim.github.io/PyDI/_modules/PyDI/utils/profiler.html
[*] Crawling: https://wbsg-uni-mannheim.github.io/PyDI/index.html
[*] Crawling: https://wbsg-uni-mannheim.github.io/PyDI/_sources/index.rst.txt
[*] Crawling: https://wbsg-uni-mannheim.github.io/PyDI/_modules/PyDI/utils/llm.html
[*] Crawling: https://wbsg-uni-mannheim.github.io/PyDI/_modules/PyDI/utils/cluster_stats.html
[*] Crawling: https://wbsg-uni-mannheim.github.io/PyDI/_modules/PyDI/schemamatching/translator.html
[*] Crawling: https://wbsg-uni-mannheim.github.io/PyDI/api/PyDI.schemamatching.html
[*] Crawling: https://wbsg-uni

## Creating Vector DB

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.embeddings import SentenceTransformerEmbeddings
from langchain.vectorstores import Chroma

DB_DIR = "./pydi_apidocs_vector_db"


splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,
    chunk_overlap=200,
)

chunks = []
for doc in docs:
    chunks.extend(splitter.split_text(doc))

print(f"Total chunks: {len(chunks)}")

embeddings = SentenceTransformerEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

db = Chroma.from_texts(
    texts=chunks,
    embedding=embeddings,
    persist_directory=DB_DIR
)

db.persist()
print("[+] PyDI documentation vector DB built")


Total chunks: 1476


/var/folders/lx/mstfnlnx381664y7plq189hm0000gn/T/ipykernel_91402/3728183112.py:22: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = SentenceTransformerEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")


[+] PyDI documentation vector DB built


In [11]:
def load_reference_db():
    embeddings = SentenceTransformerEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")
    # embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

    return Chroma(
        persist_directory=DB_DIR,
        embedding_function=embeddings,
    )


def query_pydi_reference(query: str) -> str:
    db = load_reference_db()
    results = db.similarity_search(query, k=3)

    if not results:
        return "No relevant reference content found."

    output = "\n\n-----\n\n".join([r.page_content for r in results])
    return output.replace('\n', ' ')

In [12]:
import textwrap

textwrap.wrap(query_pydi_reference('how to implement standard blocker?'))

['. strip () return text or None def _instantiate_blocker ( blocker_cfg',
 ': Dict [ str , Any ], df_left : pd . DataFrame , df_right : pd .',
 'DataFrame , id_column : str , ) -> Tuple [ str , BaseBlocker ]:',
 '"""Instantiate a blocker from configuration.""" blocker_type =',
 'blocker_cfg . get ( "type" ) name = blocker_cfg . get ( "name" ) or',
 'blocker_type params = blocker_cfg . get ( "params" , {}) if',
 'blocker_type == "standard" : on = params . get ( "on" ) or [] if not',
 'on : raise ValueError ( "Standard blocker configuration requires \'on\'',
 'columns" ) blocker = StandardBlocker ( df_left , df_right , on = on ,',
 'id_column = id_column , batch_size = params . get ( "batch_size" ,',
 '100_000 ), output_dir = params . get ( "output_dir" , "output" ),',
 'preprocess = params . get ( "preprocess" ), ) return name or',
 '"StandardBlocker" , blocker if blocker_type == "sorted_neighbourhood"',
 ': key = params . get ( "key" ) window = params . get ( "window" , 3 )',
 'if not 